In [0]:
--===========================================
-- QUANTOS PEDIDOS FORAM REALIZADOS NO TOTAL
--===========================================
SELECT
  COUNT(*) AS numero_pedidos,
  COUNT(DISTINCT order_id) AS pedidos_unicos
FROM
  sales_project.olist_ecommerce.olist_orders_dataset



In [0]:
--=====================================
-- QUANTOS CLIENTES EXISTEM POR ESTADO?
--=====================================

SELECT * FROM sales_project.olist_ecommerce.olist_customers_dataset;

SELECT
  customer_state,
  COUNT(DISTINCT customer_unique_id) AS numero_clientes  
FROM
  sales_project.olist_ecommerce.olist_customers_dataset
GROUP BY
  customer_state
ORDER BY
  numero_clientes DESC

In [0]:
--=====================================
-- QUANTOS PEDIDOS CADA CLIENTE FEZ?
--=====================================

SELECT
  customer_unique_id AS cliente,
  COUNT(order_id) AS numero_pedidos
FROM
  sales_project.olist_ecommerce.olist_customers_dataset
LEFT JOIN
  sales_project.olist_ecommerce.olist_orders_dataset
  ON
    sales_project.olist_ecommerce.olist_customers_dataset.customer_id = sales_project.olist_ecommerce.olist_orders_dataset.customer_id
GROUP BY
  cliente
ORDER BY
  numero_pedidos DESC


In [0]:
--========================================
-- QUAL VALOR TOTAL VENDIDO PELA EMPRESA?
--========================================

SELECT * FROM sales_project.olist_ecommerce.olist_order_items_dataset;

SELECT
  COUNT(seller_id) AS numero_itens_vendidos,
  FORMAT_NUMBER(SUM(price), 2) AS total_vendido
FROM
  sales_project.olist_ecommerce.olist_order_items_dataset

In [0]:
--===============================================
-- QUANTOS PRODUTOS FORAM VENDIDOS POR CATEGORIA
--===============================================

SELECT * FROM sales_project.olist_ecommerce.olist_order_items_dataset;
SELECT * FROM sales_project.olist_ecommerce.olist_products_dataset;

SELECT DISTINCT(product_category_name) FROM sales_project.olist_ecommerce.olist_products_dataset;


SELECT
  COUNT(sales_project.olist_ecommerce.olist_order_items_dataset.product_id) AS numero_de_produtos,
  product_category_name AS categoria_produto
FROM
  sales_project.olist_ecommerce.olist_order_items_dataset
INNER JOIN
  sales_project.olist_ecommerce.olist_products_dataset
    ON
      sales_project.olist_ecommerce.olist_order_items_dataset.product_id = sales_project.olist_ecommerce.olist_products_dataset.product_id
GROUP BY
  categoria_produto
ORDER BY
  numero_de_produtos DESC

-- APÓS ANALISE VERIFIQUE QUE ALGUNS PRODUTOS ESTÃO SEM CATEGORIA, O QUE ACENDE A LANTERNA PARA QUEM UTILIZA A INFORMAÇÃO DE VERIFICAR POR QUE OS PRODUTOS ESTÃO SEM CATEGORIA NA BASE--



In [0]:
--===============================================
-- QUAIS PEDIDOS TIVERAM MAIOR VALOR TOTAL?
--===============================================

SELECT * FROM sales_project.olist_ecommerce.olist_orders_dataset;
SELECT * FROM sales_project.olist_ecommerce.olist_order_items_dataset;

SELECT
  sales_project.olist_ecommerce.olist_orders_dataset.order_id AS pedidos,
  SUM(price) AS valor_total
FROM
  sales_project.olist_ecommerce.olist_orders_dataset
INNER JOIN
  sales_project.olist_ecommerce.olist_order_items_dataset
    ON
      sales_project.olist_ecommerce.olist_orders_dataset.order_id = sales_project.olist_ecommerce.olist_order_items_dataset.order_id
GROUP BY 
  pedidos
ORDER BY
  valor_total DESC
LIMIT 10

In [0]:
--===============================================
-- QUAIS ESTADOS GERAM MAIS RECEITA?
--===============================================

SELECT * FROM sales_project.olist_ecommerce.olist_customers_dataset;
SELECT * FROM sales_project.olist_ecommerce.olist_orders_dataset;
SELECT * FROM sales_project.olist_ecommerce.olist_order_items_dataset;

SELECT
  customer_state AS estado,
  SUM(price) AS valor_total
FROM
  sales_project.olist_ecommerce.olist_order_items_dataset
INNER JOIN
  sales_project.olist_ecommerce.olist_orders_dataset
    ON
      sales_project.olist_ecommerce.olist_order_items_dataset.order_id = sales_project.olist_ecommerce.olist_orders_dataset.order_id
      INNER JOIN
        sales_project.olist_ecommerce.olist_customers_dataset
          ON
            sales_project.olist_ecommerce.olist_orders_dataset.customer_id = sales_project.olist_ecommerce.olist_customers_dataset.customer_id
GROUP BY
  estado
ORDER BY
  valor_total DESC


In [0]:
--===============================================
-- QUAIS CLIENTES TEM GASTO TOTAL ACIMA DA MÉDIA?
--================================================

SELECT * FROM sales_project.olist_ecommerce.olist_customers_dataset;
SELECT * FROM sales_project.olist_ecommerce.olist_orders_dataset;
SELECT * FROM sales_project.olist_ecommerce.olist_order_items_dataset;

SELECT
  sales_project.olist_ecommerce.olist_customers_dataset.customer_id AS consumidor,
  SUM(price) AS gasto
FROM
  sales_project.olist_ecommerce.olist_customers_dataset
INNER JOIN
  sales_project.olist_ecommerce.olist_orders_dataset
    ON
      sales_project.olist_ecommerce.olist_customers_dataset.customer_id = sales_project.olist_ecommerce.olist_orders_dataset.customer_id
        INNER JOIN
          sales_project.olist_ecommerce.olist_order_items_dataset
            ON
              sales_project.olist_ecommerce.olist_orders_dataset.order_id = sales_project.olist_ecommerce.olist_order_items_dataset.order_id
GROUP BY
  consumidor
HAVING SUM(price) > (
  SELECT
    AVG(gasto_por_cliente)
  FROM (
    SELECT 
      sales_project.olist_ecommerce.olist_customers_dataset.customer_id,
      SUM(price) AS gasto_por_cliente
    FROM 
      sales_project.olist_ecommerce.olist_customers_dataset
    INNER JOIN
      sales_project.olist_ecommerce.olist_orders_dataset
      ON 
        sales_project.olist_ecommerce.olist_customers_dataset.customer_id = sales_project.olist_ecommerce.olist_orders_dataset.customer_id
        INNER JOIN 
          sales_project.olist_ecommerce.olist_order_items_dataset
          ON 
            sales_project.olist_ecommerce.olist_orders_dataset.order_id = sales_project.olist_ecommerce.olist_order_items_dataset.order_id
            GROUP BY
              sales_project.olist_ecommerce.olist_customers_dataset.customer_id
  )
  
);


In [0]:
--==============================================
-- QUEM SÃO OS 5 CLIENTES QUE MAIS GASTARAM?
--==============================================

WITH gasto_clientes AS (
  SELECT
    olist_customers_dataset.customer_id,
    SUM(price) AS gasto
  FROM 
    sales_project.olist_ecommerce.olist_customers_dataset
  INNER JOIN
    sales_project.olist_ecommerce.olist_orders_dataset
    ON
      sales_project.olist_ecommerce.olist_customers_dataset.customer_id = sales_project.olist_ecommerce.olist_orders_dataset.customer_id
        INNER JOIN
          sales_project.olist_ecommerce.olist_order_items_dataset
            ON
              sales_project.olist_ecommerce.olist_orders_dataset.order_id = sales_project.olist_ecommerce.olist_order_items_dataset.order_id
              GROUP BY
                olist_customers_dataset.customer_id
)

SELECT * FROM gasto_clientes
ORDER BY gasto DESC
LIMIT 5